<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/ParlorTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parlor test: how well do detection and tracking work on the entry-parlor camera?

The parlor camera (fixed, overhead, wide-angle) looks down on the parlor entry: cows walk in fast and stand **shoulder to shoulder** in herringbone stalls on both sides. This notebook runs the calving pipeline's steps on one parlor video to see how they cope, before building anything parlor-specific:

| Cell | What | Model | Look at |
|---|---|---|---|
| 3 | 01 find where the cows move (the recorder saves a new picture only on change), cut the **busiest 5 minutes**, rebuilt from the AVI chunks (a plain copy of these AVIs is empty) | own AVI reader + ffmpeg | busiest minutes; length 5:00, 30 fps |
| 4 | **First look**: the clip's details + 12 sample frames with YOLO's boxes | YOLO26m (bigger than the calving YOLO26n: crowded scene) | `FirstLook.jpg`: is every cow boxed? one box per cow? |
| 5 | 02 detect, 5 frames per second (cows move fast) | YOLO26m | `CowShare`, cows per frame |
| 6 | 03 track every cow | SAM 2.1 Small, video mode | `PreviewTracks.mp4`: does each cow keep one colour while walking in and standing? |
| 7 | 04 masked crops | SAM 2.1 Small | `Sheets/Overview.jpg`: one cow per crop, not two neighbours |

Outputs go to `ParlorPipelineOutputs/<video>/` (separate from the calving folders). Start with a **5-minute window** (cell 2); if the preview looks good, widen it. Needs a **T4 GPU**. No pool and no cow IDs yet: those come from the parlor stall list (BoviSync) in the next step. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install the model libraries.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/Bovi-analytics/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline import PrepareClips, ProbeVideo
from CalvingPipeline.Parlor import FirstLook
from CalvingPipeline.AviFrames import AviActivity, AviStats, BusiestWindow
from CalvingPipeline.Detect import DetectCows
from CalvingPipeline.Track import TrackCows
from CalvingPipeline.Crop import CropCows
from CalvingPipeline.Paths import FindVideo, ParlorOutputRoot
import pandas as pd
from IPython.display import Image, Video, display
def ShowVideo(Path_):
    """Play a preview inside the notebook (open bigger files from Drive)."""
    if Path_.exists() and Path_.stat().st_size < 50 * 2**20:
        display(Video(str(Path_), embed=True, width=1000))
    print("Preview:", Path_, "" if Path_.exists() else "(not made: no frames)")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoFile = "EntryParlor09_11Nov.avi"   # file name in EntryParlorVideos (found also in its October / November subfolders)
VideoStem = "EntryParlor09Nov11"        # output folder name
ClipStart, ClipEnd = "Busiest", 5   # "Busiest" = the 5 minutes with the most movement (cell 3 finds them);
                                    # or times, e.g. "00:00:00", "00:05:00"
Rebuild = True            # rebuild timestamps (the parlor AVIs need it); "Auto" = only if a plain copy is broken

DetModel = "yolo26m.pt"   # medium YOLO: better than nano for packed, overlapping cows
ImageSize = 1024          # picture size for YOLO
Confidence = 0.25         # lowest YOLO confidence kept
TargetFps = 5             # frames per second looked at (cows walk in fast; calving uses 2)
SeedConfidence = 0.4      # YOLO confidence to start following a new cow
ChunkSeconds = 60         # SAM restarts its memory every minute (many cows = more memory)
Half = False              # True = half precision, faster (check the preview)
MinFrames, MinConfidence = 10, 0.4   # crops: drop tracks shorter than 2 s
CropsPerTrack, CropEverySeconds = 6, 5

VideoDir = ParlorOutputRoot / VideoStem
ClipPath = VideoDir / "ReferenceClip.mp4"
VideoPath = FindVideo(VideoFile)
print("Video:", VideoPath, "\nOutputs:", VideoDir)

In [ ]:
# 3. Step 01 · Cut the window. The parlor AVIs have broken timestamps (a plain copy comes out 0:00 long),
#    so the clip is re-encoded with rebuilt timestamps (Rebuild in cell 2).
# The recorder only saves a new picture when something changes: find where the cows move.
Activity = AviActivity(VideoPath, TicksPerSecond=ProbeVideo(VideoPath)["Fps"] or 150)
VideoDir.mkdir(parents=True, exist_ok=True)
Activity.to_csv(VideoDir / "Activity.csv", index=False)
print(f"{Activity.RealFrames.sum()} real frames over {len(Activity)} minutes; busiest minutes:")
display(Activity.sort_values("RealFrames", ascending=False).head(10))
if ClipStart == "Busiest":
    Start, End = BusiestWindow(Activity, Minutes=ClipEnd)
    print(f"Busiest {ClipEnd} minutes: {Activity.Time.iloc[Start // 60]} to {Activity.Time.iloc[min(End // 60, len(Activity) - 1)]}")
else:
    Start, End = ClipStart, ClipEnd
Summary = PrepareClips(VideoPath, VideoDir, Start, End, Rebuild=Rebuild)
Clip = Summary["ReferenceClip"]
print(f"Clip {Summary['ClipStart']}-{Summary['ClipEnd']}: {Clip['Duration']}, {Clip['Fps']} fps, {Clip['Width']}x{Clip['Height']}")
assert Clip["DurationSeconds"] > 1, "The clip is empty: set Rebuild = True in cell 2 and rerun cells 2-3"

In [ ]:
# 4. First look: the clip's details and 12 sample frames with YOLO's boxes (about a minute).
Look = FirstLook(ClipPath, VideoDir / "FirstLook", Samples=12, ModelName=DetModel, Confidence=Confidence, ImageSize=ImageSize)
display(Look)
display(Image(str(VideoDir / "FirstLook" / "FirstLook.jpg"), width=1400))

In [ ]:
# 5. Step 02 · Detect cows (YOLO26m) at 5 fps.
DetectCows(ClipPath, VideoDir / "Step02Detect", 0, None, TargetFps=TargetFps, ModelName=DetModel,
           Confidence=Confidence, ImageSize=ImageSize, Preview=True)
Det = pd.read_csv(VideoDir / "Step02Detect" / "Detections.csv")
PerFrame = Det.groupby("TimeSeconds").size()
print(f"cows per frame: median {PerFrame.median():.0f}, max {PerFrame.max()}; "
      f"confidence median {Det.Confidence.median():.2f}")
display(pd.read_csv(VideoDir / "Step02Detect" / "DetectionSummary.csv"))
ShowVideo(VideoDir / "Step02Detect" / "PreviewDetections.mp4")   # YOLO: a box on every cow, 4x speed

In [ ]:
# 6. Step 03 · Track every cow (SAM 2.1, video mode). Rerun to continue a stopped run.
TrackCows(ClipPath, VideoDir / "Step02Detect", VideoDir / "Step03Track", ChunkSeconds=ChunkSeconds,
          SeedConfidence=SeedConfidence, Half=Half, Preview=True)
Summary = pd.read_csv(VideoDir / "Step03Track" / "TrackSummary.csv")
display(Summary)
print(f"{len(Summary)} tracks; median length {(Summary.LastSeconds - Summary.FirstSeconds).median():.0f} s")
ShowVideo(VideoDir / "Step03Track" / "PreviewTracks.mp4")   # SAM: one colour per cow, 4x speed

In [ ]:
# 7. Step 04 · Masked crops of every cow.
CropCows(ClipPath, VideoDir / "Step03Track" / "Tracks.csv", VideoDir / "Step04Crop", MinFrames=MinFrames,
         MinConfidence=MinConfidence, CropsPerTrack=CropsPerTrack, CropEverySeconds=CropEverySeconds)
display(pd.read_csv(VideoDir / "Step04Crop" / "KeptTracks.csv"))
if (VideoDir / "Step04Crop" / "Sheets" / "Overview.jpg").exists():
    display(Image(str(VideoDir / "Step04Crop" / "Sheets" / "Overview.jpg")))
else:
    print("No crops: step 03 found no tracks (check cells 4-6).")

## What to look for (and what comes next)

- **First look / step 02:** every cow boxed once (not two cows in one box, not one cow in two boxes); boxes on the far side of the frame and on cows walking in fast. `MedianBoxWidth` in pixels tells how small the cows are (small = coarse crops).
- **Step 03 preview:** each cow keeps one colour from walking in to standing in her stall. Colours jumping between neighbours = SAM mixes cows; then tracking is only usable while they stand still.
- **Step 04 overview:** one cow per crop, coat pattern visible.

**Next (parlor-specific):** match cows to the BoviSync stall list by **stall position** while they stand (stall 1–28 right side, 29–56 left side), which needs only the standing period, instead of by entry order (one broken track would shift every later rank). Then fingerprints (DINOv3, step 05) with real farm IDs, and the calving-vs-parlor match.